# PriorityLab — M2 Frozen MiniLM Embeddings + LogReg

**Goal:** encode emails with frozen `all-MiniLM-L6-v2`, train Logistic Regression on vectors, compare against M1 TF-IDF on the **same splits**.

**Prereq:** M0 go/no-go done; optionally run M1 first for comparison numbers.

**Install (once):**
```bash
pip install -r requirements-m2.txt
```

First run downloads the MiniLM weights (~80MB). Encoder is **not** fine-tuned.

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.data import load_prepared_splits
from src.embed import DEFAULT_MODEL_NAME, encode_texts, load_encoder, save_embeddings
from src.evaluate import evaluate_model, plot_confusion_matrix, save_metrics
from src.train import fit_baselines
from src.train_embeddings import MODEL_VERSION, fit_embedding_classifier, save_embedding_model

REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
MODELS = ROOT / "models"
CACHE = ROOT / "models" / "embedding_cache"
FIGURES.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
print("encoder", DEFAULT_MODEL_NAME)
print("ROOT", ROOT)

## 1. Same splits as M1

In [ ]:
train_df, val_df, test_df = load_prepared_splits()
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, part.shape, "important rate", round(part["important"].mean(), 4))

X_train_text, y_train = train_df["text"], train_df["important"]
X_val_text, y_val = val_df["text"], val_df["important"]
X_test_text, y_test = test_df["text"], test_df["important"]

## 2. Encode with frozen MiniLM

Embeddings are features only — the transformer weights are not updated.

In [ ]:
encoder = load_encoder()
X_train = encode_texts(X_train_text, encoder=encoder)
X_val = encode_texts(X_val_text, encoder=encoder)
X_test = encode_texts(X_test_text, encoder=encoder)
print("train embeddings", X_train.shape)

save_embeddings(CACHE / "train.npz", X_train)
save_embeddings(CACHE / "val.npz", X_val)
save_embeddings(CACHE / "test.npz", X_test)
print("cached under", CACHE)

## 3. Fit linear head on embeddings

In [ ]:
emb_clf = fit_embedding_classifier(X_train, y_train)
val_emb = evaluate_model(emb_clf, X_val, y_val)
print("MiniLM+LogReg validation")
print("accuracy", round(val_emb["accuracy"], 4), "macro_f1", round(val_emb["macro_f1"], 4))
print("recall_important", round(val_emb["recall_important"], 4))
print(val_emb["classification_report"])

## 4. Side-by-side vs TF-IDF (same train/val/test)

In [ ]:
tfidf_models = fit_baselines(X_train_text, y_train)
tfidf_clf = tfidf_models["tfidf_logreg"]
val_tfidf = evaluate_model(tfidf_clf, X_val_text, y_val)

print("Validation comparison")
print(f"{'model':20s} {'acc':>8s} {'macro_f1':>10s} {'recall_imp':>12s}")
for name, m in [("tfidf_logreg", val_tfidf), ("minilm_logreg", val_emb)]:
    print(
        f"{name:20s} {m['accuracy']:8.4f} {m['macro_f1']:10.4f} {m['recall_important']:12.4f}"
    )

## 5. Test fold + save M2 artifacts

In [ ]:
test_emb = evaluate_model(emb_clf, X_test, y_test)
test_tfidf = evaluate_model(tfidf_clf, X_test_text, y_test)
print("Test MiniLM+LogReg")
print(test_emb["classification_report"])

cm_path = plot_confusion_matrix(
    test_emb["confusion_matrix"],
    FIGURES / "m2_minilm_logreg_confusion.png",
    title="MiniLM + LogReg (test)",
)
metrics_path = save_metrics(
    {
        "model_version": MODEL_VERSION,
        "encoder": DEFAULT_MODEL_NAME,
        "encoder_frozen": True,
        "validation": {
            "tfidf_logreg": {k: v for k, v in val_tfidf.items() if k != "classification_report"},
            "minilm_logreg": {k: v for k, v in val_emb.items() if k != "classification_report"},
        },
        "test": {
            "tfidf_logreg": {k: v for k, v in test_tfidf.items() if k != "classification_report"},
            "minilm_logreg": {k: v for k, v in test_emb.items() if k != "classification_report"},
        },
    },
    REPORTS / "m2_metrics.json",
)
model_path = save_embedding_model(
    emb_clf,
    MODELS,
    metadata={"note": "Load encoder separately at inference; this joblib is scaler+LogReg only."},
)
print("saved", cm_path)
print("saved", metrics_path)
print("saved", model_path)

## 6. Trade-offs to document in your portfolio

- **TF-IDF + LogReg:** transparent word coefficients; weak on paraphrases.
- **Frozen MiniLM + LogReg:** denser meaning vectors; weaker lexical explanations; heavier download/deps.
- M2 does **not** fine-tune the transformer — only the linear head learns.
- Next milestone (M3): serve one chosen model behind FastAPI.